# W2.1 — Khám phá Cora

**Chủ sở hữu:** B. Phạm vi tuần 2–4, độc lập branch A.

**Input:** configs/data_protocol.json; Cora tự tải bằng Planetoid nếu chưa có cache.

Feature `[N,F]` là bag-of-words; `y` là nhãn lớp bài báo. Public masks là node split, không phải edge split.
Một cặp vô hướng tương ứng hai cột adjacency. EDA dùng full graph, chưa train encoder LP.
Nguồn: [Planetoid](https://pytorch-geometric.readthedocs.io/en/latest/generated/torch_geometric.datasets.Planetoid.html).

In [ ]:
from pathlib import Path
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p/'configs').is_dir() and (p/'notebooks').is_dir()), None)
if PROJECT_ROOT is None:
    raise RuntimeError('Hãy mở Jupyter từ repo hoặc thư mục con có configs/ và notebooks/.')

In [ ]:
get_ipython().run_line_magic('run', '"' + (PROJECT_ROOT/'notebooks/shared/runtime.ipynb').as_posix() + '"')
get_ipython().run_line_magic('run', '"' + (PROJECT_ROOT/'notebooks/shared/graph_ops.ipynb').as_posix() + '"')

D:\gnn-node-link-prediction\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


D:\gnn-node-link-prediction\.venv\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [ ]:
config = read_json(PROJECT_ROOT/'configs/data_protocol.json')
print(config)

{'dataset': 'Cora', 'public_node_split': 'public', 'seed': 42, 'undirected': True, 'val_ratio': 0.05, 'test_ratio': 0.1, 'negative_ratio': 1, 'split_id': 'cora_lp_v1', 'self_loops': 'exclude_from_pairs_add_once_per_node_in_layer', 'rounding': 'floor_val_floor_test_train_remainder'}


## Ví dụ: cạnh lặp, cạnh ngược và self-loop

In [ ]:
toy = np.array([[1,0,1,2],[0,1,0,2]],dtype=np.int64)
assert np.array_equal(canonicalize(toy,4),np.array([[0],[1]]))
print('canonical:',canonicalize(toy,4),'two directions:',bidirectional(toy,4))

canonical: [[0]
 [1]] two directions: [[0 1]
 [1 0]]


In [ ]:
data_config = read_json(PROJECT_ROOT/'configs/data_protocol.json')
data = load_cora(PROJECT_ROOT,data_config)
full_pairs = canonicalize(data.edge_index.numpy(),data.num_nodes)
fingerprints = {'features':array_hash(data.x.numpy()),'full_pairs':array_hash(full_pairs)}

In [ ]:
import matplotlib.pyplot as plt
run = new_run(PROJECT_ROOT,'w2_eda',config)
raw = data.edge_index.numpy()
degree = np.bincount(full_pairs.ravel(),minlength=data.num_nodes)
class_counts = np.bincount(data.y.numpy())
summary = {'num_nodes':data.num_nodes,'num_features':data.num_node_features,'num_classes':len(class_counts),
           'edge_columns':raw.shape[1],'undirected_pairs':full_pairs.shape[1],
           'self_loops':int((raw[0]==raw[1]).sum()),'directed_duplicates':raw.shape[1]-len(pair_set(raw)),
           'isolated_nodes':int((degree==0).sum()),'degree_min':int(degree.min()),'degree_max':int(degree.max()),
           'degree_mean':float(degree.mean()),'class_counts':class_counts.tolist(),
           'public_node_masks':{p:int(getattr(data,p+'_mask').sum()) for p in ['train','val','test']},
           'input_fingerprints':fingerprints,'feature_dtype':str(data.x.dtype),'edge_dtype':str(data.edge_index.dtype)}
assert data.x.shape[0] == data.num_nodes and np.isfinite(data.x.numpy()).all()
assert degree.sum() == 2*full_pairs.shape[1]
figdir = PROJECT_ROOT/'report/week02/figures'
figdir.mkdir(parents=True,exist_ok=True)
fig,ax = plt.subplots(figsize=(7,4)); ax.hist(degree,bins=40); ax.set(xlabel='Degree',ylabel='Node count',title='Cora: degree (undirected)'); fig.tight_layout(); fig.savefig(figdir/'degree_distribution.png',dpi=130); plt.show()
fig,ax = plt.subplots(figsize=(7,4)); ax.bar(range(len(class_counts)),class_counts); ax.set(xlabel='Class ID',ylabel='Node count',title='Cora: class distribution'); fig.tight_layout(); fig.savefig(figdir/'class_distribution.png',dpi=130); plt.show()
result = publish_result(PROJECT_ROOT,run,'results/week02/eda_summary.json',summary,config,2,'W2.1','eda','Cora')
print(summary); print('run_id:',run.name)

{'num_nodes': 2708, 'num_features': 1433, 'num_classes': 7, 'edge_columns': 10556, 'undirected_pairs': 5278, 'self_loops': 0, 'directed_duplicates': 0, 'isolated_nodes': 0, 'degree_min': 1, 'degree_max': 168, 'degree_mean': 3.8980797636632203, 'class_counts': [351, 217, 418, 818, 426, 298, 180], 'public_node_masks': {'train': 140, 'val': 500, 'test': 1000}, 'input_fingerprints': {'features': '3a607980290f1de036161b9a3965260ca662fc485f7e0dbedbd973e7ea97dd22', 'full_pairs': '26e18992954ff41d67a8486ecbf4e38bb0e40f2836502656045cd3e8a600ca00'}, 'feature_dtype': 'torch.float32', 'edge_dtype': 'torch.int64'}
run_id: 20260928T070055_w2_eda_9ff70574


## Kết luận và tự kiểm tra

Chỉ các assertion chạy thành công mới tạo kết quả bàn giao. Output bên trên chứa đường dẫn/run ID để truy vết. B chưa được đánh dấu đã tự review.

1. Vì sao số cột cạnh khác số cặp vô hướng?
2. Public masks dùng cho nhiệm vụ nào?
3. Node ID có bị đổi sau canonicalize không?